# Huấn luyện mô hình gốc DeepVQE với bộ VCTK-DEMAND (Pure PyTorch)
Notebook này thiết lập môi trường để tải bộ dữ liệu, liên kết Google Drive, clone mã nguồn DeepVQE, xử lý metadata (file csv) và huấn luyện mô hình.

**Không cần SpeechBrain** — toàn bộ logic training viết bằng Pure PyTorch để đảm bảo tương thích mọi môi trường.

## 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
!pip install torchaudio tensorboard soundfile pandas tqdm einops

from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/DeepVQE_Workspace'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

## 2. Clone mã nguồn DeepVQE

In [ ]:
# Repo này có sẵn deepvqe.py, ablation/ và stream/ benchmark scripts.
GIT_REPO = "https://github.com/hoxuanphu/Pdeepvqe.git"
REPO_DIR = "deepvqe"

if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}
else:
    print(f"Thư mục {REPO_DIR} đã tồn tại.")
    if not os.path.exists(os.path.join(REPO_DIR, "ablation")):
        print("Lưu ý: repo hiện có chưa có ablation/; cell benchmark sẽ tự lấy từ Pdeepvqe nếu cần.")

os.chdir(REPO_DIR)
print(f"Đã vào thư mục mã nguồn: {os.getcwd()}")

## 3. Tải bộ dữ liệu VoiceBank-DEMAND
Dữ liệu gốc được tải từ Đại học Edinburgh (chiếm khoảng 4GB - 5GB sau khi giải nén).

**Lưu ý**: VCTK-DEMAND gốc có sample rate 48kHz. Code training sẽ tự resample xuống 16kHz.

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Thư mục chứa file ZIP trên Google Drive
drive_data_dir = "/content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand"
os.makedirs(drive_data_dir, exist_ok=True)

# Thư mục trên máy ảo Colab (Local SSD) để giải nén và train cho nhanh
data_dir = "/content/data/voicebank-demand"
os.makedirs(data_dir, exist_ok=True)

datasets = [
    ("clean_trainset_28spk_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip"),
    ("noisy_trainset_28spk_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip"),
    ("clean_testset_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip"),
    ("noisy_testset_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip")
]

for filename, url in datasets:
    drive_zip = os.path.join(drive_data_dir, filename)
    local_zip = os.path.join(data_dir, filename)
    
    # 1. Tải về Google Drive nếu chưa có
    if not os.path.exists(drive_zip):
        print(f"Đang tải {filename} về Google Drive...")
        os.system(f"wget -q --show-progress {url} -O {drive_zip}")
    
    # 2. Copy từ Google Drive sang Local SSD
    if not os.path.exists(local_zip):
        print(f"Đang copy {filename} từ Drive sang Local SSD...")
        shutil.copy2(drive_zip, local_zip)
    
    # 3. Giải nén trên Local SSD
    extract_folder = local_zip.replace(".zip", "")
    if not os.path.exists(extract_folder):
        print(f"Đang giải nén {filename}...")
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        print(f"Hoàn tất giải nén {filename}.")

print("Dữ liệu đã sẵn sàng trên Local SSD của Colab!")


## 4. Xử lý và phân chia Dataset (Tạo file CSV)
Tạo các file metadata dạng CSV (`train.csv`, `valid.csv`, `test.csv`).

In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    
    assert len(clean_files) == len(noisy_files), f"Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})"
    
    data = []
    for c, n in zip(clean_files, noisy_files):
        filename = os.path.basename(c)
        data.append({
            "ID": filename.replace(".wav", ""),
            "clean_wav": os.path.abspath(c),
            "noisy_wav": os.path.abspath(n)
        })
        
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Đã tạo {output_csv} với {len(df)} mẫu.")

base_dir = data_dir
# Tạo file CSV thô
create_csv(f"{base_dir}/clean_trainset_28spk_wav", f"{base_dir}/noisy_trainset_28spk_wav", f"{base_dir}/train_full.csv")
create_csv(f"{base_dir}/clean_testset_wav", f"{base_dir}/noisy_testset_wav", f"{base_dir}/test.csv")

# Tách train_full thành train (90%) và valid (10%)
df_train_full = pd.read_csv(f"{base_dir}/train_full.csv")
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(f"{base_dir}/train.csv", index=False)
df_train_full.iloc[split_idx:].to_csv(f"{base_dir}/valid.csv", index=False)

print(f"Train: {split_idx} | Valid: {len(df_train_full) - split_idx}")
print("Hoàn tất tạo metadata (train.csv, valid.csv, test.csv)!")

## 5. Cấu hình Hyperparameters

In [ ]:
CONFIG = {
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    
    'batch_size': 4,
    'num_workers': 2,
    'epochs': 100,
    'lr': 1e-3,          # Cập nhật theo repo gốc
    'grad_clip': 5.0,
    
    # Loss weights
    'lamda_ri': 30,
    'lamda_mag': 70,
    'compress_factor': 0.3,
    
    'train_csv': f'{data_dir}/train.csv',
    'valid_csv': f'{data_dir}/valid.csv',
    'test_csv': f'{data_dir}/test.csv',
    
    'output_dir': '/content/drive/MyDrive/DeepVQE_Workspace/checkpoints/deepvqe_vctk',
    
    # Scheduler
    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    
    # Early stopping
    'early_stopping_patience': 15,
}

print('Config:', CONFIG)


## 6. Dataset & DataLoader (Pure PyTorch)

In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG['seed'])


class VCTKDemandDataset(Dataset):
    """Dataset đơn giản cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV."""
    
    def __init__(self, csv_path, sample_rate=16000, segment_len=None):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len  # Số sample cắt mỗi đoạn (None = giữ nguyên)
    
    def __len__(self):
        return len(self.df)
    
    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)  # [T]
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        
        # Cắt về cùng chiều dài
        min_len = min(noisy.shape[0], clean.shape[0])
        noisy = noisy[:min_len]
        clean = clean[:min_len]
        
        # Random crop nếu đặt segment_len
        if self.segment_len is not None and min_len > self.segment_len:
            start = random.randint(0, min_len - self.segment_len)
            noisy = noisy[start:start + self.segment_len]
            clean = clean[start:start + self.segment_len]
        
        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    """Pad tất cả wav trong batch về cùng chiều dài (max len)."""
    max_len = max(item['noisy'].shape[0] for item in batch)
    
    noisy_batch = []
    clean_batch = []
    ids = []
    for item in batch:
        n, c = item['noisy'], item['clean']
        pad_len = max_len - n.shape[0]
        if pad_len > 0:
            n = torch.nn.functional.pad(n, (0, pad_len))
            c = torch.nn.functional.pad(c, (0, pad_len))
        noisy_batch.append(n)
        clean_batch.append(c)
        ids.append(item['id'])
    
    return {
        'noisy': torch.stack(noisy_batch),   # [B, T]
        'clean': torch.stack(clean_batch),   # [B, T]
        'id': ids,
    }


# Tạo DataLoader
# Dùng segment_len = 3s * 16000 = 48000 sample cho training để tiết kiệm VRAM
train_dataset = VCTKDemandDataset(CONFIG['train_csv'], CONFIG['sample_rate'], segment_len=48000)
valid_dataset = VCTKDemandDataset(CONFIG['valid_csv'], CONFIG['sample_rate'], segment_len=None)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=False, collate_fn=collate_fn
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')

## 7. Khởi tạo Model, Optimizer, Scheduler

In [ ]:
import sys
import time
import json
from pathlib import Path

# Đảm bảo import được deepvqe.py
work_dir = '/content/drive/MyDrive/DeepVQE_Workspace/deepvqe'
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)

from deepvqe import DeepVQE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Tạo model
model = DeepVQE().to(device)

# Bật DataParallel nếu có nhiều GPU
if torch.cuda.device_count() > 1:
    print(f'Bật DataParallel trên {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M')

# Optimizer & Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'],
    min_lr=CONFIG['scheduler_min_lr'],
)

# STFT window
window = torch.hann_window(CONFIG['win_length']).to(device)

# Output dir
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f'Output dir: {output_dir}')

## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log

In [ ]:
import csv

def make_stft(wav, n_fft, hop_length, win_length, win):
    """wav [B, T] -> spec [B, F, T_frames, 2]"""
    spec = torch.stft(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length,
                      window=win, return_complex=True)
    return torch.view_as_real(spec)

def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    """spec [B, F, T_frames, 2] -> wav [B, T]"""
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(complex_spec, n_fft=n_fft, hop_length=hop_length,
                       win_length=win_length, window=win, length=length)

def compute_loss(model, noisy_wav, clean_wav, cfg, win):
    """Tính HybridLoss giống repo SEtrain: Compressed RI + Compressed Mag + negative SI-SNR."""
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']
    
    device = noisy_wav.device
    
    # Forward pass
    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)
    pred_stft = model(noisy_spec)
    
    # Cắt padding cho độ dài phù hợp
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)
    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]
    
    # Tách Real / Imaginary
    pred_stft_real, pred_stft_imag = pred_stft[:,:,:,0], pred_stft[:,:,:,1]
    true_stft_real, true_stft_imag = true_stft[:,:,:,0], true_stft[:,:,:,1]
    
    # Tính Magnitude
    pred_mag = torch.sqrt(pred_stft_real**2 + pred_stft_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_stft_real**2 + true_stft_imag**2 + 1e-12)
    
    # Phổ nén (Compressed Spectrum)
    pred_real_c = pred_stft_real / (pred_mag**(1 - c))
    pred_imag_c = pred_stft_imag / (pred_mag**(1 - c))
    true_real_c = true_stft_real / (true_mag**(1 - c))
    true_imag_c = true_stft_imag / (true_mag**(1 - c))
    
    # RI Loss & Mag Loss
    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)
    
    # ISTFT để tính SI-SNR
    # Cửa sổ lúc này có lũy thừa 0.5 theo repo gốc, nhưng ta có thể dùng cửa sổ hann bình thường để đồng bộ
    y_pred = torch.istft(pred_stft_real + 1j*pred_stft_imag, n_fft=n_fft, hop_length=hop, win_length=win_len, window=win)
    y_true = torch.istft(true_stft_real + 1j*true_stft_imag, n_fft=n_fft, hop_length=hop, win_length=win_len, window=win)
    
    # Đảm bảo cùng độ dài
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]
    
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (torch.sum(torch.square(y_true), dim=-1, keepdim=True) + 1e-8)
    sisnr = -torch.log10(torch.sum(torch.square(y_target), dim=-1, keepdim=True) / torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True) + 1e-8).mean()
    
    # Tổng Loss = 30 * RI + 70 * Mag + SISNR
    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    
    return loss, {'loss': loss.item(), 'ri_loss': (real_loss + imag_loss).item(), 'mag_loss': mag_loss.item(), 'sisnr': sisnr.item()}

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None):
    """★ V2: Lưu thêm scaler state cho AMP resume. (Safe Drive Patch)"""
    import os, shutil
    from pathlib import Path
    
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    ckpt = {
        'model': model_state,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_loss': best_loss,
        'bad_epochs': bad_epochs,
    }
    if scaler is not None:
        ckpt['scaler'] = scaler.state_dict()
        
    base_name = os.path.basename(str(path))
    if os.path.exists('/content'):
        temp_local = f'/content/temp_{base_name}'
    elif os.path.exists('/kaggle/working'):
        temp_local = f'/kaggle/working/temp_{base_name}'
    else:
        temp_local = f'temp_{base_name}'
        
    # 1. Lưu vào SSD cục bộ trước
    torch.save(ckpt, temp_local)
    
    # 2. Copy sang một file tạm trên Drive (.tmp)
    temp_drive = str(path) + '.tmp'
    shutil.copy2(temp_local, temp_drive)
    
    # 3. Ghi đè file cũ bằng file tạm một cách an toàn (atomic rename)
    # Nếu bị ngắt kết nối giữa chừng lúc copy, file gốc vẫn còn nguyên vẹn.
    try:
        os.replace(temp_drive, str(path))
    except OSError:
        # Dự phòng cho một số hệ thống file không hỗ trợ replace trực tiếp
        if os.path.exists(str(path)):
            os.remove(str(path))
        os.rename(temp_drive, str(path))
        
    # 4. Xóa file cục bộ
    if os.path.exists(temp_local):
        os.remove(temp_local)

def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu'):
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    target = model.module if hasattr(model, 'module') else model
    target.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler and ckpt.get('scheduler'):
        scheduler.load_state_dict(ckpt['scheduler'])
    return ckpt.get('epoch', 0), ckpt.get('best_loss'), ckpt.get('bad_epochs', 0)

def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

print('Hàm tiện ích đã sẵn sàng.')


## 9. Vòng lặp huấn luyện (Training Loop)
- Auto-resume từ `last.pt` nếu có.
- Ghi log mỗi epoch vào `train_log.csv`.

In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# === Auto-resume nếu có checkpoint ===
start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / 'last.pt'
if resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path, model, optimizer, scheduler, device=device
    )
    print(f'Đã resume từ epoch {start_epoch}, best_loss={best_loss:.6f}, bad_epochs={bad_epochs}')
else:
    print('Bắt đầu training từ đầu.')

log_path = output_dir / 'train_log.csv'
print(f'Log file: {log_path}')


# === TRAINING LOOP ===
for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    # --- TRAIN ---
    model.train()
    train_losses = []
    t0 = time.time()
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for batch_idx, batch in enumerate(pbar):
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)
        
        optimizer.zero_grad()
        loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
        loss.backward()
        
        if CONFIG['grad_clip']:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        
        optimizer.step()
        train_losses.append(parts)
        pbar.set_postfix({'loss': f"{parts['loss']:.4f}"})
    
    avg_train = {k: np.mean([d[k] for d in train_losses]) for k in train_losses[0]}
    elapsed = time.time() - t0
    
    # --- VALID ---
    model.eval()
    valid_losses = []
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
            noisy = batch['noisy'].to(device)
            clean = batch['clean'].to(device)
            _, parts = compute_loss(model, noisy, clean, CONFIG, window)
            valid_losses.append(parts)
    
    avg_valid = {k: np.mean([d[k] for d in valid_losses]) for k in valid_losses[0]}
    
    # --- Scheduler ---
    scheduler.step(avg_valid['loss'])
    current_lr = optimizer.param_groups[0]['lr']
    
    # --- Logging ---
    print(
        f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
        f"Train Loss: {avg_train['loss']:.6f} (ri={avg_train['ri_loss']:.4f}, mag={avg_train['mag_loss']:.4f}, sisnr={avg_train['sisnr']:.4f}) | "
        f"Valid Loss: {avg_valid['loss']:.6f} (ri={avg_valid['ri_loss']:.4f}, mag={avg_valid['mag_loss']:.4f}, sisnr={avg_valid['sisnr']:.4f}) | "
        f"LR: {current_lr:.2e} | Time: {elapsed:.0f}s"
    )
    
    # --- Ghi log vào CSV ---
    append_log(log_path, {
        'epoch': epoch,
        'train_loss': f"{avg_train['loss']:.6f}",
        'train_ri_loss': f"{avg_train['ri_loss']:.6f}",
        'train_mag_loss': f"{avg_train['mag_loss']:.6f}",
        'train_sisnr': f"{avg_train['sisnr']:.6f}",
        'valid_loss': f"{avg_valid['loss']:.6f}",
        'valid_ri_loss': f"{avg_valid['ri_loss']:.6f}",
        'valid_mag_loss': f"{avg_valid['mag_loss']:.6f}",
        'valid_sisnr': f"{avg_valid['sisnr']:.6f}",
        'lr': f"{current_lr:.2e}",
        'time_s': f"{elapsed:.0f}",
    })
    
    # --- Checkpoint ---
    is_best = best_loss is None or avg_valid['loss'] < best_loss
    if is_best:
        best_loss = avg_valid['loss']
        bad_epochs = 0
    else:
        bad_epochs += 1
    
    # Luôn lưu last.pt
    save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs)
    
    # Lưu best.pt nếu tốt hơn
    if is_best:
        save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs)
        print(f'  >>> Saved best model (valid_loss={best_loss:.6f})')
    
    # Early stopping
    if bad_epochs >= CONFIG['early_stopping_patience']:
        print(f'Early stopping sau {bad_epochs} epoch không cải thiện. Best loss: {best_loss:.6f}')
        break

print(f'\n=== Huấn luyện hoàn tất! ===')
print(f'Best valid loss: {best_loss:.6f}')
print(f'Checkpoint: {output_dir}')


## 10. Kiểm tra nhanh sau training
Chạy inference trên một mẫu test để xác nhận model hoạt động.

In [ ]:
import IPython.display as ipd

# Load best model
best_ckpt = output_dir / 'best.pt'
if best_ckpt.exists():
    load_checkpoint(best_ckpt, model, device=device)
    print('Đã load best.pt')

model.eval()
test_df = pd.read_csv(CONFIG['test_csv'])
sample = test_df.iloc[0]

sig_noisy, sr = torchaudio.load(sample['noisy_wav'])
if sr != CONFIG['sample_rate']:
    sig_noisy = torchaudio.functional.resample(sig_noisy, sr, CONFIG['sample_rate'])

with torch.no_grad():
    noisy_wav = sig_noisy.squeeze(0).to(device)
    spec = make_stft(noisy_wav.unsqueeze(0), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
    # Xử lý DataParallel
    m = model.module if hasattr(model, 'module') else model
    pred = m(spec)
    enhanced = make_istft(pred, CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy_wav.shape[0])

print(f'Sample: {sample["ID"]}')
print('Noisy:')
ipd.display(ipd.Audio(sig_noisy.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))
print('Enhanced:')
ipd.display(ipd.Audio(enhanced.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))

sig_clean, sr_c = torchaudio.load(sample['clean_wav'])
if sr_c != CONFIG['sample_rate']:
    sig_clean = torchaudio.functional.resample(sig_clean, sr_c, CONFIG['sample_rate'])
print('Clean (ground truth):')
ipd.display(ipd.Audio(sig_clean.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))

## 11. Chạy ablation benchmarks trên Colab
Cell này chạy smoke checks, benchmark kiến trúc cho các cấu hình trong `ablation/`, và có thể bật thêm train/eval/ONNX khi cần.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
WORKSPACE = Path('/content/drive/MyDrive/DeepVQE_Workspace')
AB_SOURCE_DIR = None  # ví dụ: '/content/drive/MyDrive/deepvqe/ablation'; None = tự tìm/clone Pdeepvqe
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/content/Pdeepvqe_benchmark_src')

# Nếu muốn chạy ít cấu hình để smoke test trước, sửa thành: ['Baseline', 'B1a']
AB_CONFIGS = [
    'Baseline', 'B1a', 'B1b', 'B2', 'B3',
    'C1', 'C1a-g2', 'C1a-g4', 'C1b', 'C2', 'C3', 'C4',
]

RUN_SMOKE_TESTS = True
RUN_ARCH_BENCHMARK = True

# None = tự tìm best.pt trong Drive/Colab. Nếu có nhiều best.pt, điền path cụ thể vào đây.
BASELINE_CHECKPOINT = None

# Hai phần dưới tốn GPU lâu hơn. Bật True khi muốn benchmark chất lượng sau train.
RUN_TRAINING = False
RUN_QUALITY_EVAL = True
RUN_ONNX_EXPORT = False

AB_EPOCHS = 1
AB_BATCH_SIZE = 8
AB_NUM_WORKERS = 0
AB_PIN_MEMORY = False
AB_EARLY_STOP_PATIENCE = 12
AB_EARLY_STOP_MIN_DELTA = 0.01
AB_EARLY_STOP_MIN_EPOCHS = 20
AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AB_USE_DATA_PARALLEL = torch.cuda.is_available() and torch.cuda.device_count() > 1
RESUME_EXISTING = True

ARCH_FRAMES = 63
ARCH_WARMUP = 1
ARCH_REPEATS = 3
INSTALL_OPTIONAL_PACKAGES = False  # ptflops / pesq / pystoi / onnxruntime nếu cần
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = WORKSPACE / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Device: {AB_DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA GPUs: {torch.cuda.device_count()}')
    for gpu_idx in range(torch.cuda.device_count()):
        print(f'  GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'DataParallel: {AB_USE_DATA_PARALLEL}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source(explicit_dir=None):
    if explicit_dir:
        path = Path(explicit_dir)
        if not (path / 'deepvqe_ablation.py').exists():
            raise FileNotFoundError(f'AB_SOURCE_DIR không hợp lệ: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source(AB_SOURCE_DIR)
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        'ckpt = torch.load(str(checkpoint_path), map_location=device)',
        "ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)",
    )
    text = text.replace(
        'model.load_state_dict(ckpt["model"])',
        "state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))\n"
        "    state = {k.replace('module.', '', 1): v for k, v in state.items()}\n"
        "    model.load_state_dict(state, strict=True)",
    )
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Đã patch {path} để load checkpoint upload an toàn hơn.')

patch_eval_script_for_uploaded_checkpoints()

results_dir = WORKSPACE / 'results/ablation'
runs_dir = WORKSPACE / 'runs/ablation'
onnx_dir = WORKSPACE / 'onnx_models/ablation'
manifest_dir = WORKSPACE / 'metadata/ablation_manifests'
for path in [results_dir, runs_dir, onnx_dir, manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

arch_csv = results_dir / 'ablation_arch_benchmark.csv'
quality_csv = results_dir / 'ablation_quality.csv'
onnx_csv = results_dir / 'ablation_onnx.csv'
summary_csv = results_dir / 'ablation_summary.csv'

def find_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy BASELINE_CHECKPOINT: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob('best.pt'))
        if candidates:
            break
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

uploaded_baseline_ckpt = find_best_checkpoint(BASELINE_CHECKPOINT)
if uploaded_baseline_ckpt is not None:
    print(f'Dùng Baseline checkpoint: {uploaded_baseline_ckpt}')
else:
    print('Không tìm thấy best.pt trong Drive/Colab; Baseline eval/ONNX sẽ dùng checkpoint trong runs_dir nếu có.')

def checkpoint_for_config(config_id):
    if config_id == 'Baseline' and uploaded_baseline_ckpt is not None:
        return uploaded_baseline_ckpt
    return runs_dir / config_id / 'best.pt'

if INSTALL_OPTIONAL_PACKAGES:
    packages = ['ptflops']
    if RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    run_py(['-m', 'pip', 'install', '-q', *packages])

def csv_to_jsonl(csv_path, jsonl_path):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path

metadata_base = Path(globals().get('data_dir', str(WORKSPACE / 'data/voicebank-demand')))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    for split, csv_path in csv_manifests.items():
        jsonl_manifests[split] = csv_to_jsonl(csv_path, manifest_dir / f'{split}.jsonl')
    print('Đã tạo manifest JSONL cho ablation:')
    for split, path in jsonl_manifests.items():
        print(f'  {split}: {path}')
elif RUN_TRAINING or RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')
else:
    print('Chưa có metadata CSV; bỏ qua manifest train/eval và chỉ chạy benchmark kiến trúc.')

if RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *AB_CONFIGS])

if RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', *AB_CONFIGS,
        '--device', AB_DEVICE,
        '--frames', ARCH_FRAMES,
        '--warmup', ARCH_WARMUP,
        '--repeats', ARCH_REPEATS,
    ])

if RUN_TRAINING:
    for config_id in AB_CONFIGS:
        ab_output_dir = runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', ab_output_dir,
            '--device', AB_DEVICE,
            '--epochs', AB_EPOCHS,
            '--batch-size', AB_BATCH_SIZE,
            '--num-workers', AB_NUM_WORKERS,
            '--early-stop-patience', AB_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', AB_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', AB_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = ab_output_dir / 'last.pt'
        if not AB_PIN_MEMORY:
            args += ['--no-pin-memory']
        if AB_USE_DATA_PARALLEL:
            args += ['--data-parallel']
        if RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt, '--ignore-bad-resume']
        run_py(args)

if RUN_QUALITY_EVAL:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', quality_csv,
            '--device', AB_DEVICE,
        ])

if RUN_ONNX_EXPORT:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', onnx_dir,
            '--results', onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

archive_base = Path('/content/ablation_benchmark_results')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
print(f'Đã nén kết quả benchmark: {archive_path}')
display(FileLink(str(archive_path)))